In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.26.0',
    'python-dotenv>=1.0.0',
    'pyyaml>=6.0',
    'requests>=2.32.0',
    'soundfile>=0.12.1',
    'numpy>=1.26.0',
    'pyloudnorm>=0.1.1',
    'torch>=2.3.0',
    'torchaudio>=2.3.0',
    'faster-whisper>=1.0.0',
], check=True)
subprocess.run(['apt-get', 'install', '-qq', '-y', 'ffmpeg'], check=True)

In [ ]:
import os
import re
import json
import time
import threading
import subprocess
import sys
from pathlib import Path
from datetime import datetime

import yaml
import requests
import numpy as np
import soundfile as sf
import pyloudnorm as pyln
import torch
from huggingface_hub import HfApi

WORK_DIR        = Path('/kaggle/working')
STANDARD_DIR    = WORK_DIR / 'standardized'
SEGMENTS_DIR    = WORK_DIR / 'segments'
SNR_FLAG_DIR    = WORK_DIR / 'snr_flagged'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p1c.json'
CONFIG_DIR      = Path('/kaggle/input/S2S-pipline-v2-0-2/config')

SEGMENTS_DIR.mkdir(parents=True, exist_ok=True)
SNR_FLAG_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SR       = 24000
TARGET_LUFS     = -23.0
MIN_SEG_SEC     = 3.0
MAX_SEG_SEC     = 15.0
CHUNK_MINUTES   = 10
SNR_PASS        = 20.0
SNR_FLAG        = 15.0
WHISPER_CONF    = 0.75
URDU_CHARS      = set('ابپتثجچحخدذرزژسشصضطظعغفقکگلمنوہھیئاآءۃے')
SAVE_EVERY      = 10

In [ ]:
def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        c = UserSecretsClient()
        secrets = {
            'HF_TOKEN_PRIMARY':   c.get_secret('HF_TOKEN_PRIMARY'),
            'HF_TOKEN_SECONDARY': c.get_secret('HF_TOKEN_SECONDARY'),
            'HF_TOKEN_TERTIARY':  c.get_secret('HF_TOKEN_TERTIARY'),
            'ANTHROPIC_API_KEY':  c.get_secret('ANTHROPIC_API_KEY'),
        }
        print('[secrets] loaded from Kaggle Secrets')
        return secrets
    except Exception:
        pass

    env_file = Path('.env')
    if env_file.exists():
        from dotenv import load_dotenv
        load_dotenv(env_file)
        print('[secrets] loaded from .env')

    required = ['HF_TOKEN_PRIMARY', 'HF_TOKEN_SECONDARY', 'HF_TOKEN_TERTIARY', 'ANTHROPIC_API_KEY']
    missing = [k for k in required if not os.environ.get(k)]
    if missing:
        raise RuntimeError(f'Missing secrets: {missing}')
    return {k: os.environ[k] for k in required}

SECRETS     = load_secrets()
HF_TOKEN    = SECRETS['HF_TOKEN_PRIMARY']

with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)

STAGE0_REPO = repos_cfg['repos']['stage0_codec']['repo_id']
HF_API      = HfApi(token=HF_TOKEN)
print(f'[config] stage0 repo: {STAGE0_REPO}')

In [ ]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f:
                state = json.load(f)
            print(f'[checkpoint] local — done={len(state["done"])} segments={state["stats"]["total_segments"]}')
            return state
        except Exception:
            pass

    try:
        url = f'https://huggingface.co/datasets/{STAGE0_REPO}/resolve/main/checkpoint_p1c.json'
        r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=30)
        if r.status_code == 200:
            state = r.json()
            with open(CHECKPOINT_PATH, 'w') as f:
                json.dump(state, f)
            print(f'[checkpoint] HF fallback — done={len(state["done"])}')
            return state
    except Exception:
        pass

    print('[checkpoint] fresh start')
    return {
        'done': [],
        'stats': {
            'total_segments': 0,
            'snr_pass': 0,
            'snr_flag': 0,
            'snr_reject': 0,
            'lang_reject': 0,
            'loudness_reject': 0,
        },
        'last_updated': None,
    }


cp_lock = threading.Lock()

def save_checkpoint(state, upload=False):
    with cp_lock:
        state['last_updated'] = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
        tmp = str(CHECKPOINT_PATH) + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(state, f)
        os.replace(tmp, str(CHECKPOINT_PATH))

    if not upload:
        return
    for attempt in range(6):
        try:
            HF_API.upload_file(
                path_or_fileobj=json.dumps(state).encode(),
                path_in_repo='checkpoint_p1c.json',
                repo_id=STAGE0_REPO,
                repo_type='dataset',
                commit_message='p1c checkpoint',
            )
            return
        except Exception as e:
            time.sleep(min(2 ** attempt, 60))


state    = load_checkpoint()
done_set = set(state['done'])

In [ ]:
vad_model_cache = {}

def get_vad_model():
    if 'model' not in vad_model_cache:
        model, utils = torch.hub.load(
            repo_or_dir='snakers4/silero-vad',
            model='silero_vad',
            force_reload=False,
            onnx=False,
            verbose=False,
        )
        vad_model_cache['model'] = model
        vad_model_cache['get_ts'] = utils[0]
        print('[vad] Silero VAD loaded')
    return vad_model_cache['model'], vad_model_cache['get_ts']


whisper_cache = {}

def get_whisper(size='tiny', device='cpu'):
    key = (size, device)
    if key not in whisper_cache:
        from faster_whisper import WhisperModel
        compute = 'float16' if device == 'cuda' else 'int8'
        whisper_cache[key] = WhisperModel(size, device=device, compute_type=compute)
        print(f'[whisper] {size} loaded on {device}')
    return whisper_cache[key]


def compute_snr(audio, frame_length=2048, noise_percentile=10):
    if len(audio) < frame_length:
        return 0.0
    hop = frame_length // 2
    energies = np.array([
        np.mean(audio[i:i + frame_length] ** 2)
        for i in range(0, len(audio) - frame_length, hop)
    ])
    energies = energies[energies > 0]
    if len(energies) == 0:
        return 0.0
    noise_floor = np.percentile(energies, noise_percentile)
    if noise_floor <= 0:
        return 60.0
    return float(10 * np.log10(np.mean(energies) / noise_floor))


def snr_label(snr_db):
    if snr_db >= SNR_PASS:
        return 'pass'
    if snr_db >= SNR_FLAG:
        return 'flag'
    return 'reject'


def normalize_loudness(audio, sr):
    audio_f64 = audio.astype(np.float64)
    meter = pyln.Meter(sr)
    loudness = meter.integrated_loudness(audio_f64)
    if not np.isfinite(loudness):
        return None
    normalized = pyln.normalize.loudness(audio_f64, loudness, TARGET_LUFS)
    if np.max(np.abs(normalized)) > 1.0:
        return None
    return normalized.astype(np.float32)


def detect_language_chunk(audio_path):
    model = get_whisper('tiny', 'cpu')
    _, info = model.transcribe(str(audio_path), language=None, task='transcribe', beam_size=1)
    lang = info.language
    prob = round(info.language_probability, 3)
    if lang == 'hi' and prob < 0.85:
        lang = 'ur'
    return lang, prob


def run_vad_on_chunk(audio_array):
    model, get_ts = get_vad_model()
    tensor = torch.FloatTensor(audio_array)
    raw = get_ts(tensor, model, sampling_rate=TARGET_SR)
    segments = []
    for ts in raw:
        start = ts['start'] / TARGET_SR
        end   = ts['end']   / TARGET_SR
        dur   = end - start
        if dur < MIN_SEG_SEC:
            continue
        if dur <= MAX_SEG_SEC:
            segments.append({'start': start, 'end': end, 'duration': round(dur, 3)})
            continue
        cursor = start
        while cursor < end:
            seg_end = min(cursor + MAX_SEG_SEC, end)
            seg_dur = seg_end - cursor
            if seg_dur >= MIN_SEG_SEC:
                segments.append({'start': cursor, 'end': seg_end, 'duration': round(seg_dur, 3)})
            cursor += MAX_SEG_SEC
    return segments

In [ ]:
manifest_local = WORK_DIR / 'video_manifest.jsonl'
video_meta     = {}

if manifest_local.exists():
    with open(manifest_local, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                v = json.loads(line)
                video_meta[v['video_id']] = v

wav_files = sorted(STANDARD_DIR.glob('*.wav'))
pending   = [p for p in wav_files if p.stem not in done_set]

print(f'[clean_cpu] {len(wav_files)} total WAVs, {len(done_set)} already done, {len(pending)} to process')

get_vad_model()
get_whisper('tiny', 'cpu')

In [ ]:
seg_records = []

for file_idx, wav_path in enumerate(pending):
    vid_id   = wav_path.stem
    vid_info = video_meta.get(vid_id, {})
    file_segs_written = 0

    try:
        with sf.SoundFile(str(wav_path)) as sf_file:
            total_samples  = len(sf_file)
            sr             = sf_file.samplerate
            total_duration = total_samples / sr

        chunk_samples  = CHUNK_MINUTES * 60 * TARGET_SR
        chunk_offset   = 0
        chunk_idx      = 0

        while chunk_offset < total_samples:
            with sf.SoundFile(str(wav_path)) as sf_file:
                sf_file.seek(chunk_offset)
                chunk = sf_file.read(chunk_samples, dtype='float32')

            if len(chunk) == 0:
                break

            chunk_offset_sec = chunk_offset / TARGET_SR
            vad_segs         = run_vad_on_chunk(chunk)

            for seg in vad_segs:
                abs_start = chunk_offset_sec + seg['start']
                abs_end   = chunk_offset_sec + seg['end']

                start_idx = int(seg['start'] * TARGET_SR)
                end_idx   = int(seg['end']   * TARGET_SR)
                seg_audio = chunk[start_idx:end_idx]

                snr_db  = compute_snr(seg_audio)
                s_label = snr_label(snr_db)

                with cp_lock:
                    state['stats']['total_segments'] += 1
                    state['stats'][f'snr_{s_label}'] += 1

                if s_label == 'reject':
                    continue

                normalized = normalize_loudness(seg_audio, TARGET_SR)
                if normalized is None:
                    with cp_lock:
                        state['stats']['loudness_reject'] += 1
                    continue

                seg_id   = f'{vid_id}_c{chunk_idx:03d}_s{file_segs_written:04d}'
                seg_file = SEGMENTS_DIR / f'{seg_id}.wav' if s_label == 'pass' else SNR_FLAG_DIR / f'{seg_id}.wav'

                sf.write(str(seg_file), normalized, TARGET_SR, subtype='PCM_16')

                seg_records.append({
                    'seg_id':          seg_id,
                    'video_id':        vid_id,
                    'video_title':     vid_info.get('title', ''),
                    'channel_id':      vid_info.get('channel_id', ''),
                    'channel_category': vid_info.get('category', 'general'),
                    'query_used':      vid_info.get('query_used', ''),
                    'abs_start_sec':   round(abs_start, 3),
                    'abs_end_sec':     round(abs_end, 3),
                    'duration_sec':    round(seg['duration'], 3),
                    'snr_db':          round(snr_db, 2),
                    'snr_label':       s_label,
                    'seg_path':        str(seg_file),
                })

                file_segs_written += 1

            chunk_offset += chunk_samples
            chunk_idx    += 1

        with cp_lock:
            if vid_id not in done_set:
                done_set.add(vid_id)
                state['done'].append(vid_id)

    except Exception as e:
        print(f'  [error] {vid_id}: {e}')

    if (file_idx + 1) % SAVE_EVERY == 0 or file_idx + 1 == len(pending):
        upload_now = (file_idx + 1) % (SAVE_EVERY * 5) == 0
        save_checkpoint(state, upload=upload_now)
        print(f'  [{file_idx+1}/{len(pending)}] segs={state["stats"]["total_segments"]} '
              f'pass={state["stats"]["snr_pass"]} flag={state["stats"]["snr_flag"]} '
              f'reject={state["stats"]["snr_reject"]}')

In [ ]:
seg_records_path = WORK_DIR / 'seg_records_pre_lang.jsonl'
with open(seg_records_path, 'w', encoding='utf-8') as f:
    for rec in seg_records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

print(f'\n[clean_cpu] summary')
print(f'  total segments   : {state["stats"]["total_segments"]}')
print(f'  snr pass         : {state["stats"]["snr_pass"]}')
print(f'  snr flag (demucs): {state["stats"]["snr_flag"]}')
print(f'  snr reject       : {state["stats"]["snr_reject"]}')
print(f'  loudness reject  : {state["stats"]["loudness_reject"]}')
print(f'  records written  : {len(seg_records)}')
print(f'  pass segments dir: {SEGMENTS_DIR}')
print(f'  flag segments dir: {SNR_FLAG_DIR}')

save_checkpoint(state, upload=True)
print('\n[done] ready for p1d_clean_gpu.ipynb')